In [0]:
EH_NAMESPACE = '<namespace>.servicebus.windows.net'
EH_NAME = '<event-hub-name>'
SERVICE_CREDENTIAL = '<service-credential>'

CATALOG = spark.sql('SELECT current_catalog()').first()[0]
SCHEMA = 'iss_tracker'

BRONZE = f'{SCHEMA}.bronze_events'
SILVER = f'{SCHEMA}.silver_positions'
GOLD = f'{SCHEMA}.gold_positions'

CKPT_ROOT = f'/Volumes/{CATALOG}/{SCHEMA}/pipeline_state/checkpoints'
CKPT_BRONZE = f'{CKPT_ROOT}/bronze_v1'
CKPT_SILVER = f'{CKPT_ROOT}/silver_v2'

KAFKA_OPTIONS = {
    'kafka.bootstrap.servers': f'{EH_NAMESPACE}:9093',
    'subscribe': EH_NAME,
    'databricks.serviceCredential': SERVICE_CREDENTIAL
}

print('CATALOG    :', CATALOG)
print('Hub        :', EH_NAMESPACE, '/', EH_NAME)
print('Checkpoints:', CKPT_ROOT)

In [0]:
from pyspark.sql.functions import col

df = (spark.read.format('kafka')
      .options(**KAFKA_OPTIONS)
      .option('startingOffsets', 'earliest')
      .option('endingOffsets', 'latest')
      .load())

print('Events in the hub: ', df.count())

(df.select(col('partition'), col('offset'), col('timestamp').alias('broker_time'), col('value').cast('string').alias('raw_json'))
 .orderBy(col('offset').desc())
 .show(5, truncate=90))